In [97]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [98]:
df = sns.load_dataset('titanic')
target = 'survived'

In [99]:
X = df.drop(columns=[target])
Y = df[target]

In [100]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,random_state=42,test_size=0.2,stratify = Y)

In [101]:
X_train['embark_town'].value_counts()

,count
embark_town,
Southampton,516
Cherbourg,139
Queenstown,55


In [102]:
# remove redundant columns
X_train = X_train.drop(columns=['class','embark_town'])
X_test = X_test.drop(columns=['class','embark_town'])

In [103]:
num_columns = X_train.select_dtypes(include = np.number).columns.tolist()
cat_columns = X_train.select_dtypes(exclude = np.number).columns.tolist()
num_columns


['pclass', 'age', 'sibsp', 'parch', 'fare']

In [104]:
X_train['deck'] = X_train['deck'].fillna(X_train['deck'].mode()[0])

In [105]:
# Split into binary, ordinal category & nominal categories
for col in cat_columns:
  print(col, X_train[col].unique())
  print("_"*50)



sex ['male' 'female']
__________________________________________________
embarked ['S' 'C' 'Q' nan]
__________________________________________________
who ['man' 'woman' 'child']
__________________________________________________
adult_male [ True False]
__________________________________________________
deck ['C', 'A', 'D', 'E', 'F', 'B', 'G']
Categories (7, object): ['A', 'B', 'C', 'D', 'E', 'F', 'G']
__________________________________________________
alive ['yes' 'no']
__________________________________________________
alone [ True False]
__________________________________________________


In [106]:
binary_and_ordinal_cols = ['sex','adult_male',"alone",'alive']
multi_class_nominal_cols = ['embarked', 'who', 'deck']

In [107]:
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

In [108]:
# Label encoders
labe_encoders = {}
for col in binary_and_ordinal_cols:
  le = LabelEncoder()
  le.fit(X_train[col])
  labe_encoders[col] = le.classes_

  X_train_enc[col] = le.transform(X_train[col])
  X_test_enc[col] = le.transform(X_test[col])
print(labe_encoders)

{'sex': array(['female', 'male'], dtype=object), 'adult_male': array([False,  True]), 'alone': array([False,  True]), 'alive': array(['no', 'yes'], dtype=object)}


In [109]:
# one hot encoder
enc = OneHotEncoder(drop='first',handle_unknown='ignore', sparse_output=False)
enc.fit(X_train[multi_class_nominal_cols])


OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

In [110]:
train_ohe = enc.transform(X_train[multi_class_nominal_cols])
test_ohe = enc.transform(X_test[multi_class_nominal_cols])

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [111]:
ohe_cols = enc.get_feature_names_out(multi_class_nominal_cols)
print("onHotEncoder columns:", ohe_cols)

onHotEncoder columns: ['embarked_Q' 'embarked_S' 'embarked_nan' 'who_man' 'who_woman' 'deck_B'
 'deck_C' 'deck_D' 'deck_E' 'deck_F' 'deck_G']


In [112]:
train_ohe_df = pd.DataFrame(train_ohe, columns = ohe_cols, index = X_train.index)
test_ohe_df = pd.DataFrame(test_ohe, columns = ohe_cols, index = X_test.index)

In [113]:
X_train_enc = X_train_enc.drop(columns = multi_class_nominal_cols)
X_test_enc = X_test_enc.drop(columns = multi_class_nominal_cols)

In [115]:
X_train_enc = pd.concat([X_train_enc, train_ohe_df], axis=1)
X_test_enc = pd.concat([X_train_enc,test_ohe_df], axis = 1)

X_train_enc.head()

,pclass,sex,age,sibsp,parch,fare,adult_male,alive,alone,embarked_Q,embarked_S,embarked_nan,who_man,who_woman,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G,embarked_Q,embarked_S,embarked_nan,who_man,who_woman,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G
692,3,1,NaN,0,0,56.4958,1,1,1,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
481,2,1,NaN,0,0,0.0000,1,0,1,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
527,1,1,NaN,0,0,221.7792,1,0,1,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
855,3,0,18.0,0,1,9.3500,0,1,0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
801,2,0,31.0,1,1,26.2500,0,1,0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
